# Retry Logic Notebook
Implement and test a robust retry wrapper with exponential backoff.

In [ ]:
# Install dependencies
%pip install -q requests tenacity

## Configure Retry Parameters
Define maximum attempts, backoff factor, jitter, and retryable exceptions.

In [ ]:
from typing import Iterable, Type
import requests
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type, before_sleep_log
import logging

# Configure logging
logging.basicConfig(level=logging.INFO, format='[%(asctime)s] %(levelname)s %(message)s')
logger = logging.getLogger("retry-demo")

MAX_ATTEMPTS = 5
BACKOFF_BASE = 0.5  # seconds
BACKOFF_MULTIPLIER = 2  # exponential factor
BACKOFF_MAX = 8.0  # cap
RETRY_EXCEPTIONS: Iterable[Type[BaseException]] = (requests.exceptions.RequestException,)

logger.info("Retry parameters loaded")

## Implement Retry Wrapper
Reusable decorator with exponential backoff, jitter, and logging.

In [ ]:
def retryable(
    attempts: int = MAX_ATTEMPTS,
    base: float = BACKOFF_BASE,
    multiplier: float = BACKOFF_MULTIPLIER,
    maximum: float = BACKOFF_MAX,
    exceptions: Iterable[Type[BaseException]] = RETRY_EXCEPTIONS,
):
    """Decorator to add retry with exponential backoff."""
    def wrapper(fn):
        return retry(
            reraise=True,
            stop=stop_after_attempt(attempts),
            wait=wait_exponential(multiplier=multiplier, min=base, max=maximum),
            retry=retry_if_exception_type(exceptions),
            before_sleep=before_sleep_log(logger, logging.WARNING),
        )(fn)

    return wrapper


@retryable()
def flaky_operation(counter={"i": 0}):
    counter["i"] += 1
    if counter["i"] < 3:
        raise requests.exceptions.ConnectionError("Simulated transient failure")
    return "success"


logger.info("Retry wrapper ready (flaky_operation will fail twice then succeed)")

## Test Retry Logic with Mock
Simulate transient failures and ensure retries occur as expected.

In [ ]:
import unittest
from unittest.mock import Mock

class TestRetry(unittest.TestCase):
    def test_retries_then_succeeds(self):
        fn = Mock(side_effect=[requests.exceptions.Timeout(), requests.exceptions.Timeout(), "ok"])

        @retryable(attempts=3)
        def wrapped():
            return fn()

        result = wrapped()
        self.assertEqual(result, "ok")
        self.assertEqual(fn.call_count, 3)

suite = unittest.TestLoader().loadTestsFromTestCase(TestRetry)
unittest.TextTestRunner(verbosity=2).run(suite)

## Integrate Retry with HTTP Requests
Wrap requests.get with retry logic and demonstrate transient failure handling.

In [ ]:
session = requests.Session()

@retryable()
def get_with_retry(url: str):
    resp = session.get(url, timeout=2)
    resp.raise_for_status()
    return resp.json() if 'application/json' in resp.headers.get('Content-Type','') else resp.text

try:
    data = get_with_retry("https://httpbin.org/status/503")
except Exception as e:
    logger.warning(f"Expected failure after retries: {e}")

# Successful call
ok = get_with_retry("https://httpbin.org/get")
print("Success keys:", list(ok.keys()))